# EAD Model - Exposure at Default

For term loans, EAD is simply the current outstanding balance, since there is no undrawn credit to consider. For revolving loans, EAD accounts for the tendency of borrowers approaching default to draw down more of their available credit limit, using a Credit Conversion Factor (CCF) applied to the undrawn portion: EAD = current_balance + (CCF x undrawn_amount). A CCF of 0.50 is used here, a commonly cited industry rule-of-thumb assumption for unsecured revolving retail credit in the absence of historical drawdown data.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/clean_loan_data.csv')
df.shape

(10000, 17)

In [2]:
df['EAD'] = np.where(
    df['loan_type'] == 'term',
    df['current_balance'],
    np.nan  # placeholder, filled in Chunk 4
)

In [3]:
CCF = 0.50

undrawn_amount = df['credit_limit'] - df['current_balance']
revolving_ead = df['current_balance'] + (CCF * undrawn_amount)

df['EAD'] = np.where(
    df['loan_type'] == 'revolving',
    revolving_ead,
    df['EAD']  # keep the term loan values already set in Chunk 3
)

df['EAD'] = df['EAD'].round(2)

In [4]:
df.groupby('loan_type')['EAD'].describe()

df[df['loan_type']=='revolving'][['current_balance', 'credit_limit', 'EAD']].head(10)

,current_balance,credit_limit,EAD
1,17050.19,36929.54,26989.86
9,12678.60,20678.55,16678.58
12,5952.22,14500.70,10226.46
13,9579.50,17909.86,13744.68
14,12180.60,19688.23,15934.42
15,8718.44,14072.26,11395.35
21,19742.91,47843.51,33793.21
22,10539.07,22784.96,16662.02
26,5632.35,15789.12,10710.74
34,13300.81,33193.34,23247.07


In [5]:
df[['loan_id', 'loan_type', 'current_balance', 'credit_limit', 'EAD']].to_csv('../data/processed/ead_estimates.csv', index=False)
print("Saved.")

Saved.
